## Predict

In [1]:
import pandas as pd
import numpy as np
import joblib

In [2]:
final_model = joblib.load(
    "../models/final_churn_model.pkl"
)

print("Final Churn Model loaded successfully!")

Final Churn Model loaded successfully!


In [3]:
import joblib

preprocessor = joblib.load(
    "../models/preprocessor.pkl"
)

print("Preprocessor loaded successfully!")

Preprocessor loaded successfully!


In [5]:
new_customer = pd.DataFrame([{
    "gender": "Female",
    "SeniorCitizen": 0,
    "Partner": "Yes",
    "Dependents": "No",
    "tenure": 12,
    "PhoneService": "Yes",
    "MultipleLines": "No",
    "InternetService": "DSL",
    "OnlineSecurity": "No",
    "OnlineBackup": "Yes",
    "DeviceProtection": "No",
    "TechSupport": "No",
    "StreamingTV": "No",
    "StreamingMovies": "No",
    "Contract": "Month-to-month",
    "PaperlessBilling": "Yes",
    "PaymentMethod": "Electronic check",
    "MonthlyCharges": 70.0,
    "TotalCharges": 840.0
}])

new_customer

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,Female,0,Yes,No,12,Yes,No,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,70.0,840.0


In [6]:
new_customer["TenureYears"] = new_customer["tenure"] / 12

new_customer["IsNewCustomer"] = (
    new_customer["tenure"] <= 6
).astype(int)

new_customer["IsLongTermCustomer"] = (
    new_customer["tenure"] >= 24
).astype(int)

service_columns = [
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

new_customer["ServiceCount"] = (
    new_customer[service_columns] == "Yes"
).sum(axis=1)

new_customer["ExpectedTotalCharges"] = (
    new_customer["tenure"] *
    new_customer["MonthlyCharges"]
)

new_customer["ChargeDifference"] = (
    new_customer["TotalCharges"] -
    new_customer["ExpectedTotalCharges"]
)

new_customer["AverageMonthlyCharge"] = (
    new_customer["TotalCharges"] /
    new_customer["tenure"].replace(0, np.nan)
)

new_customer["AverageMonthlyCharge"] = (
    new_customer["AverageMonthlyCharge"].fillna(0)
)

new_customer

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,PaymentMethod,MonthlyCharges,TotalCharges,TenureYears,IsNewCustomer,IsLongTermCustomer,ServiceCount,ExpectedTotalCharges,ChargeDifference,AverageMonthlyCharge
0,Female,0,Yes,No,12,Yes,No,DSL,No,Yes,...,Electronic check,70.0,840.0,1.0,0,0,1,840.0,0.0,70.0


In [10]:
print("Columns expected by preprocessor:")

for transformer_name, transformer, columns in preprocessor.transformers_:
    if transformer_name != "remainder":
        print(f"\n{transformer_name}:")
        print(columns)

Columns expected by preprocessor:

num:
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'TenureYears', 'IsNewCustomer', 'IsLongTermCustomer', 'ServiceCount', 'ExpectedTotalCharges', 'ChargeDifference', 'AverageMonthlyCharge']

cat:
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'ContractTenure']


In [12]:
# ContractTenure
new_customer["ContractTenure"] = (
    new_customer["Contract"] + "_" + new_customer["tenure"].astype(str)
)

# Numerical feature engineering
new_customer["TenureYears"] = new_customer["tenure"] / 12

new_customer["IsNewCustomer"] = (
    new_customer["tenure"] <= 6
).astype(int)

new_customer["IsLongTermCustomer"] = (
    new_customer["tenure"] >= 24
).astype(int)

service_columns = [
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

new_customer["ServiceCount"] = (
    new_customer[service_columns] == "Yes"
).sum(axis=1)

new_customer["ExpectedTotalCharges"] = (
    new_customer["tenure"] *
    new_customer["MonthlyCharges"]
)

new_customer["ChargeDifference"] = (
    new_customer["TotalCharges"] -
    new_customer["ExpectedTotalCharges"]
)

new_customer["AverageMonthlyCharge"] = (
    new_customer["TotalCharges"] /
    new_customer["tenure"].replace(0, np.nan)
).fillna(0)

print("Feature engineering completed!")
new_customer

Feature engineering completed!


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,MonthlyCharges,TotalCharges,TenureYears,IsNewCustomer,IsLongTermCustomer,ServiceCount,ExpectedTotalCharges,ChargeDifference,AverageMonthlyCharge,ContractTenure
0,Female,0,Yes,No,12,Yes,No,DSL,No,Yes,...,70.0,840.0,1.0,0,0,1,840.0,0.0,70.0,Month-to-month_12


In [13]:
print("New customer columns:")
print(new_customer.columns.tolist())

New customer columns:
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'TenureYears', 'IsNewCustomer', 'IsLongTermCustomer', 'ServiceCount', 'ExpectedTotalCharges', 'ChargeDifference', 'AverageMonthlyCharge', 'ContractTenure']


In [14]:
new_customer_processed = preprocessor.transform(
    new_customer
)

print("New customer preprocessed successfully!")
print("Processed shape:", new_customer_processed.shape)

New customer preprocessed successfully!
Processed shape: (1, 48)


c:\Users\yashg\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [15] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [15]:
prediction = final_model.predict(
    new_customer_processed
)

probability = final_model.predict_proba(
    new_customer_processed
)[0, 1]

result = "Churn" if prediction[0] == 1 else "No Churn"

print("Prediction:", result)
print("Churn Probability:", f"{probability:.2%}")

Prediction: No Churn
Churn Probability: 49.74%


c:\Users\yashg\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
c:\Users\yashg\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


In [16]:
prediction = final_model.predict(
    new_customer_processed
)

probability = final_model.predict_proba(
    new_customer_processed
)[0, 1]

result = "Churn" if prediction[0] == 1 else "No Churn"

print("Prediction:", result)
print("Churn Probability:", f"{probability:.2%}")

Prediction: No Churn
Churn Probability: 49.74%


c:\Users\yashg\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
c:\Users\yashg\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


In [17]:
print("===================================")
print("       CUSTOMER CHURN PREDICTION")
print("===================================")
print(f"Prediction        : {result}")
print(f"Churn Probability : {probability:.2%}")
print("===================================")

       CUSTOMER CHURN PREDICTION
Prediction        : No Churn
Churn Probability : 49.74%
